# AI401 Assignment 1: soil grain-size distribution from photos

CNNs, ViTs and self-supervised encoders on the Kaggle competition
[Predicting Soil Grain Size Distributions from Images](https://www.kaggle.com/competitions/soil-grain-size-from-photos).

The heavy lifting (tiling, training, frozen features, DINO adaptation, interpretation, ensembles) is done by the
scripts in `scripts/` (see `remote/run_all.sh`). Every experiment writes its config, OOF predictions, test
predictions, metrics and compute usage to `results/exps/<name>/`. This notebook reads only `results/`, so it runs
without a GPU or the raw photos. The full report is in [README.md](../README.md).

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
ROOT = Path('..').resolve(); R = ROOT / 'results'
sys.path.insert(0, str(ROOT / 'src'))
from soilgs.metric import COLS, SIEVES, LOG_D, DLOG, emd, d_value, d_errors, make_valid, UNIFORM_BASELINE
pd.set_option('display.width', 200, 'display.max_columns', 30)
labels = pd.read_csv(R / 'data/labels.csv'); labels.columns = ['sample_id'] + COLS
Y = labels[COLS].to_numpy(float)

## Part 0. EDA

In [ ]:
summary = json.loads((R / 'eda/summary.json').read_text())
photos = pd.read_csv(R / 'eda/photos.csv')
print(json.dumps({k: summary[k] for k in ['n_train_samples', 'n_test_samples', 'n_photos', 'photos_per_sample', 'class_counts']}, indent=1))
display(photos.groupby(['camera']).agg(photos=('photo', 'size'), samples=('sample_id', 'nunique'), ppm=('ppm', 'first'),
        mpix=('mpix', 'median'), fov_w_mm=('field_w_mm', 'median'), fov_h_mm=('field_h_mm', 'median')).round(2))
display(pd.crosstab(photos.split, photos.camera))

In [ ]:
display(Image(R / 'figures/eda_curves.png'))
curves = pd.read_csv(R / 'eda/curves.csv')
display(curves[['sample_id', 'class', 'fines_lt_0063', 'gravel_gt_2', 'log10_d50']].sort_values('log10_d50').round(2).T)

In [ ]:
thumbs = sorted((R / 'eda/thumbs').glob('*.jpg'))
full = [t for t in thumbs if '__' not in t.name][:4]
fig, ax = plt.subplots(1, len(full), figsize=(4 * len(full), 4))
for a, t in zip(np.atleast_1d(ax), full):
    a.imshow(plt.imread(t)); a.set_title(t.stem, fontsize=8); a.axis('off')
plt.show()
t0 = full[0].stem
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, mm in zip(ax, (24, 48, 96)):
    a.imshow(plt.imread(R / f'eda/thumbs/{t0}__{mm}mm.jpg')); a.set_title(f'{mm} mm tile -> 224 px ({224/mm:.2f} px/mm)'); a.axis('off')
plt.show()

## Part 0. Metric

EMD = sum over the 10 intervals of |F_i - F^_i| * (log10 x_{i+1} - log10 x_i), in % x decades, range [0, 500].
It is the area between the two cumulative curves on a log-diameter axis, i.e. the Wasserstein-1 distance
between the two grain-size distributions measured in log10(mm).

In [ ]:
def shift_curve(c, decades):
    # the same soil seen at a wrong scale: the curve moves along log d
    return np.interp(LOG_D - decades, LOG_D, c, left=0, right=100)
mid = Y[np.argsort(d_value(Y, 50))[len(Y) // 2]]
rows = []
for k in (1.41, 2, 4):
    s = shift_curve(mid, np.log10(k))
    rows.append({'scale error': f'x{k}', 'EMD (median soil)': emd(mid, s), 'theory 100*log10(k)': 100 * np.log10(k)})
display(pd.DataFrame(rows).round(2))
print('uniform Kaggle baseline vs training curves, mean EMD:', emd(Y, UNIFORM_BASELINE[None]).mean().round(2))
print('training mean (in-sample, optimistic):', emd(Y, Y.mean(0, keepdims=True)).mean().round(2))
print('training median curve (L1-optimal per sieve):', emd(Y, np.median(Y, 0, keepdims=True)).mean().round(2))

**Strengths:** a physically meaningful distance (area between the grading curves), every decade weighs the
same, it is linear in the error so outliers do not dominate, and it is the natural loss for a cumulative target.

**Weaknesses and edge cases:**
- The grid is coarse (11 points, about 0.5 decade apart). A shift inside one interval is invisible until it crosses a sieve.
- The clay/silt end (0.002-0.063 mm) carries 1.5 of the 5 decades, but it is invisible in a photo, so a large
  part of the score is decided by what the camera cannot resolve.
- A wrong pixels-per-mm by x2 costs 30 EMD by itself (see the table), more than the gap between a trivial and a good model.
- The last point is fixed at 100, and the 63-200 mm interval contributes almost nothing for fine soils.

**Complementary metric:** errors of D10 / D30 / D50 / D60 in decades (|log10 D^ - log10 D|). Engineers use
these numbers (Cu = D60/D10, Cc = D30²/(D10·D60)), and the error reads as a factor (0.30 = factor 2).
All runs report them (`D50_err_dec`).

**Alternative target:** the 11 bin masses (mass fraction between consecutive sieves), predicted with a softmax
and accumulated. The prediction is then valid by construction (non-negative increments, ends at 100), and the
EMD can be the training loss itself. All neural models here use this head.

In [ ]:
# complementary metric on a toy example, and why isotonic repair beats a running maximum
yhat = np.array([10, 35, 30, 80, 105.])
print('running max  ', make_valid(yhat, 'max')[0], '\nisotonic PAV ', make_valid(yhat, 'isotonic')[0])
print(d_errors(Y[:3], shift_curve(Y[0], np.log10(2))[None].repeat(3, 0)))

## Part 0. Validation and tiles

- **Folds:** 6 group folds over physical samples (all photos of a sample are on the same side), stratified by
  D50 (each block of 6 samples sorted by D50 is dealt to the 6 folds). Every experiment uses the same
  `results/data/folds.csv`. Predictions are averaged tile -> photo -> sample before scoring.
- **Scale:** ppm comes from `ppm_updated.csv` scaled to the stored photo size (the Motorola Edge photos are stored at
  1600 px instead of 4000, so 11.49 -> 4.60 px/mm; Samsung A52 26.33 -> 4.55 px/mm). As a check of that table,
  the Samsung scale was also estimated without labels by matching texture spectra with the Motorola photos of the
  same soils (`scripts/calibrate.py`, figure below): 4.8 px/mm, +5%, the same bias the method shows on a
  Motorola-vs-Motorola self-check.
- **Tiles:** every photo is resampled to 4.67 px/mm, the native density of the training photos, so test iPhone
  photos are only downsampled and train photos are never upsampled. 48 mm x 48 mm tiles (224 px, 50% overlap,
  central 70% of the frame, at most 48 per photo) are cached once. 48 mm holds several medium-gravel grains, and
  0.21 mm/px still resolves coarse sand. 24 mm and 96 mm caches are kept as an ablation (Part 1).

In [ ]:
display(Image(R / 'calibration/samsung_ppm.png'))
folds = pd.read_csv(R / 'data/folds.csv')
display(folds.merge(curves[['sample_id', 'class']]).groupby('fold').agg(samples=('sample_id', 'size'),
        classes=('class', lambda s: ', '.join(sorted(s))), d50_mm=('log10_d50', lambda s: ', '.join(f'{10**v:.2g}' for v in sorted(s)))))

## Results table (all experiments)

In [ ]:
display(Markdown((R / 'tables.md').read_text()))

In [ ]:
res = pd.read_csv(R / 'results.csv')
def show(names):
    d = res.set_index('name').loc[[n for n in names if n in set(res.name)]]
    return d[['cv_mean', 'cv_std', 'D50_err_dec', 'lb_public', 'gpu_hours', 'peak_mem_gb']].round(3)

## Part 1. CNNs

In [ ]:
display(show(['base_train_mean', 'cnn_r18_adamw', 'cnn_r18_sgd', 'cnn_r18_const_nowarm', 'cnn_r18_noaug', 'cnn_r18_strongaug']))
display(show(['cnn_r18_adamw', 'cnn_r18_frozenbn', 'cnn_r18_groupnorm', 'cnn_r18_bs8_grouped', 'cnn_r18_bs8_grouped_frozenbn']))
display(show(['cnn_r18_tile24mm', 'cnn_r18_adamw', 'cnn_r18_tile96mm', 'cnn_r18_img112']))
display(show(['cnn_r18_adamw', 'cnn_r50_adamw', 'cnn_effb0_adamw', 'cnn_convnext_t_adamw', 'cnn_convnext_t_sgd']))

## Part 2. ViTs

In [ ]:
display(show(['vit_scratch', 'vit_deit_s', 'cnn_r18_adamw', 'vit_deit_s_nollrd', 'vit_deit_s_nowarm', 'vit_deit_s_sgd',
              'vit_deit_s_wd_all', 'vit_deit_s_dp0', 'vit_deit_s_img112', 'vit_deit_s_strongaug', 'vit_swin_t', 'vit_maxvit_t']))
display(Image(R / 'figures/training_curves.png'))

## Part 3. Self-supervised encoders

In [ ]:
ssl = res[res.family == 'ssl'].copy()
frz = ssl[ssl.name.str.contains('_ridge_|_knn_|_lp_')]
frz = frz.assign(encoder=frz.name.str.extract(r'^(\w+?_s(?:_da)?)_')[0], head=frz.name.str.extract(r'_(ridge|knn|lp)_')[0],
                 feats=frz.name.str.extract(r'_(?:ridge|knn|lp)_(\w+)$')[0])
display(frz.pivot_table(index=['encoder'], columns=['head', 'feats'], values='cv_mean').round(2))
display(show([n for n in res.name if n.startswith('ssl_')]).join(res.set_index('name')['cka']))

In [ ]:
for p in sorted((R / 'figures').glob('da_*_curves.png')):
    display(Image(p))

## Part 4. Interpretation and shortcut controls

In [ ]:
display(Image(R / 'figures/interpretation_maps.png'))
display(pd.read_csv(R / 'interpretation.csv').round(3))
ctrl = res[res.ctrl_shuffle.notna()].sort_values('cv_mean').groupby('family').head(2)
display(ctrl[['name', 'cv_mean', 'ctrl_shuffle', 'ctrl_grey']].round(2))
display(show(['base_uniform', 'base_train_mean', 'ctrl_camera_only', 'ctrl_colour_only']))

## Part 6. Comparison of approaches and scale robustness

In [ ]:
best = res[res.family.isin(['cnn', 'vit', 'ssl'])].sort_values('cv_mean').groupby('family').head(1)
display(best[['family', 'name', 'cv_mean', 'cv_std', 'lb_public', 'gpu_hours', 'peak_mem_gb', 'params_m', 'trainable_m',
              'scale_ref', 'res_0.5', 'res_0.25', 'ppm_x0.5', 'ppm_x2']].round(3))
display(Image(R / 'figures/scale_robustness.png'))
display(pd.read_csv(R / 'error_by_class.csv'))

## Part 7. Ensemble

In [ ]:
display(pd.read_csv(R / 'oof_error_corr.csv', index_col=0).round(2))
display(pd.read_csv(R / 'ensembles.csv'))
print(json.dumps(json.loads((R / 'final_choice.json').read_text()), indent=1))

## Appendix 2. Practical exercises (checked numerically)

In [ ]:
# 1. ViT-S/16 on 1024x1024
D, N = 384, (1024 // 16) ** 2 + 1
print('Ex1 tokens', N, '| attn matrix fp16 MB', N * N * 2 / 2**20, '| MACs/block linear', 12 * N * D**2 / 1e9, 'G, attn', 2 * N**2 * D / 1e9, 'G')
tot_big = 12 * (12 * N * D**2 + 2 * N**2 * D); n2 = 14 * 14 + 1; tot_tiles = 16 * 12 * (12 * n2 * D**2 + 2 * n2**2 * D)
print('     total 1024^2:', tot_big / 1e9, 'G MACs vs 16 tiles of 224:', tot_tiles / 1e9, 'G')
# 2. Swin stage 1
h = w = 56; C = 96; M = 7
msa, wmsa = 4*h*w*C**2 + 2*(h*w)**2*C, 4*h*w*C**2 + 2*M**2*h*w*C
print('Ex2 MSA', msa / 1e9, 'G  W-MSA', wmsa / 1e9, 'G  ratio', msa / wmsa)
# 3. weighted k-NN
s = np.array([0.9, 0.8, 0.75, 0.6]); cls = np.array(list('ABBB'))
for tau in (0.1, 1):
    wts = np.exp(s / tau); print('Ex3 tau', tau, {c: round(wts[cls == c].sum() / wts.sum(), 3) for c in 'AB'})
wts = np.exp(s / 0.1); print('    regression', (wts * np.array([60, 40, 50, 30])).sum() / wts.sum())
# 4. RankMe
sig = np.array([4, 2, 1, 1.]); p = sig / sig.sum(); print('Ex4 RankMe', np.exp(-(p * np.log(p)).sum()), '(min 1, max 4)')
# 5. linear CKA
cka = lambda X, Y: np.linalg.norm(Y.T @ X)**2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))
X = np.array([[1, 2, 3, 4.]]).T; X -= X.mean()
for Yv in ([2, 4, 6, 8.], [1, -1, 1, -1.]):
    Yc = np.array([Yv]).T; Yc -= Yc.mean(); print('Ex5 CKA', round(cka(X, Yc), 4))
# 6. valid curve
print('Ex6', make_valid(np.array([10, 35, 30, 80, 105.]), 'max')[0], make_valid(np.array([10, 35, 30, 80, 105.]), 'isotonic')[0])
# 7. EMD on a log grid
d = np.array([0.1, 1, 10]); P = np.array([20, 50, 100.]); T = np.array([40, 60, 100.])
lg = np.log10(d); print('Ex7 EMD', (np.abs(P - T)[:-1] * np.diff(lg)).sum(),
      'D50 pred', 10**np.interp(50, P, lg), 'D50 true', 10**np.interp(50, T, lg), 'x2 scale error EMD', 100 * np.log10(2))
# 8. DINO centering / sharpening
t = np.array([2, 0, 0.]); c = np.array([1, 0, -1.]); sm = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
print('Ex8', sm((t - c) / 0.5).round(4), sm(t / 0.5).round(4), 'new centre', 0.9 * c + 0.1 * t)
# 9. LoRA
r = 8; lora = 12 * (r * (384 + 1152) + r * (384 + 384)); print('Ex9 LoRA params', lora, 'share', lora / 22e6, 'full qkv', 384 * 1152)
# 10. Grad-CAM by hand
A1, A2 = np.array([[1, 0], [2, 1.]]), np.array([[0, 3], [1, 0.]]); a1, a2 = 0.5, -0.25
print('Ex10 alphas', a1, a2, 'map', np.maximum(a1 * A1 + a2 * A2, 0))